Data Ingestion


In [1]:
import pandas as pd

input_path = r"D:\spreadsheets\2019-Oct.csv"
output_path = "marts_ecommerce.csv"

print("🔧 [DE] Extracting and cleaning a 3M row slice from D drive...")

try:
    # 1. Load a safe chunk to protect your RAM
    df = pd.read_csv(input_path, nrows=3000000)
    
    # 2. Drop duplicate rows caused by web logging glitches
    df = df.drop_duplicates()

    # 3. Optimize data types (Convert text strings to actual Dates and Floats)
    df['event_time'] = pd.to_datetime(df['event_time'].str.replace(' UTC', ''))
    df['price'] = pd.to_numeric(df['price'], errors='coerce')

    # 4. Handle missing data by filling blank fields with default values
    df['brand'] = df['brand'].fillna('Unknown')
    df['category_code'] = df['category_code'].fillna('uncategorized')

    # 5. Feature Engineering: Create time attributes for easy dashboard filtering
    df['hour'] = df['event_time'].dt.hour
    df['day_of_week'] = df['event_time'].dt.day_name()

    # Save the optimized file locally
    df.to_csv(output_path, index=False)
    print(f"✅ Success! Pristine data saved locally as '{output_path}'\n")

except Exception as e:
    print(f"❌ Error during engineering phase: {e}")

🔧 [DE] Extracting and cleaning a 3M row slice from D drive...
✅ Success! Pristine data saved locally as 'marts_ecommerce.csv'



Bussiness Logic

In [2]:
import pandas as pd

print("👔 [BA] Reading clean file to extract financial metrics...")
df = pd.read_csv("marts_ecommerce.csv")

# Isolate successful purchase transactions
purchases = df[df['event_type'] == 'purchase']

# Calculate Gross Merchandise Value (GMV) and Average Order Value (AOV)
total_revenue = purchases['price'].sum()
aov = purchases['price'].mean()

print(f"\n📈 --- FINANCIAL EXECUTIVE SUMMARY ---")
print(f"• Total Gross Revenue: ${total_revenue:,.2f}")
print(f"• Average Order Value (AOV): ${aov:.2f}")

# Rank vendors by total sales volume
top_brands = purchases.groupby('brand')['price'].sum().sort_values(ascending=False).head(3)
print("\n🏅 --- TOP 3 REVENUE-DRIVING BRANDS ---")
for brand, rev in top_brands.items():
    print(f"  - {brand.upper()}: ${rev:,.2f}")

👔 [BA] Reading clean file to extract financial metrics...

📈 --- FINANCIAL EXECUTIVE SUMMARY ---
• Total Gross Revenue: $16,316,512.50
• Average Order Value (AOV): $321.19

🏅 --- TOP 3 REVENUE-DRIVING BRANDS ---
  - APPLE: $8,121,634.44
  - SAMSUNG: $3,274,733.71
  - UNKNOWN: $644,786.72


Product Analysis

In [3]:
import pandas as pd

print("🕹️ [PA] Reading clean file to analyze user behavior...")
df = pd.read_csv("marts_ecommerce.csv")

# Count total clickstream actions across the user journey
journey_counts = df['event_type'].value_counts()
views = journey_counts.get('view', 0)
carts = journey_counts.get('cart', 0)
purchases = journey_counts.get('purchase', 0)

print(f"\n🛒 --- STOREFRONT USER JOURNEY FUNNEL ---")
print(f"👀 Product Views:  {views:,}")
print(f"📥 Adds-to-Cart:   {carts:,}")
print(f"📦 Completed Paid Orders: {purchases:,}")

# Calculate conversion and drop-off rates
if views > 0:
    conv_rate = (purchases / views) * 100
    print(f"\n📊 Store Conversion Rate: {conv_rate:.2f}%")

if carts > 0:
    abandonment_rate = ((carts - purchases) / carts) * 100
    print(f"⚠️ Cart Abandonment Rate: {abandonment_rate:.2f}%")

🕹️ [PA] Reading clean file to analyze user behavior...

🛒 --- STOREFRONT USER JOURNEY FUNNEL ---
👀 Product Views:  2,904,037
📥 Adds-to-Cart:   43,728
📦 Completed Paid Orders: 50,800

📊 Store Conversion Rate: 1.75%
⚠️ Cart Abandonment Rate: -16.17%
